# Семинар 12 [17-25-2]. От Python к распределениям SciPy

Цель: самим открыть документацию [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html),
создать распределение, получить выборку и теоретические величины, сравнить их и нарисовать.

Задания идут лесенкой:

1. **Ступень 1 — по образцу.** Есть готовый пример, вы подставляете свои параметры.
2. **Ступень 2 — сами.** Выбираете и комбинируете то, что узнали; одно распределение находите в документации сами.
3. **Ступень 3 — задание с прошлого семинара.** Дописываете код с опорой на документацию.

**Перед каждой ячейкой с кодом запишите предсказание:** что она выведет или нарисует.
Потом запустите и сравните. Не совпало — это самое интересное место, разберитесь почему.

**Запуск.** В Colab загрузите скачанный `.ipynb` через «Файл → Загрузить блокнот».
Выполняйте ячейки по порядку сверху вниз (`Shift+Enter`). Если что-то сломалось —
«Среда выполнения → Перезапустить сеанс» и снова сверху вниз.

## Подготовка

Выполните обе ячейки: первая подключает библиотеки, вторая создаёт готовую функцию `plot_dist(sample, dist)` — она рисует выборку и теоретический закон на одном графике.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
# @title Готовая функция plot_dist (выполните ячейку, разбирать код не нужно)
def plot_dist(sample, dist, title=None):
    """Рисует выборку и теоретический закон на одном графике.

    sample — массив наблюдений (например, результат dist.rvs(...)),
    dist   — «замороженное» распределение, например stats.binom(n=10, p=0.3).
    """
    if not hasattr(dist, "dist"):
        raise TypeError("plot_dist ждёт распределение с параметрами, "
                        "например stats.binom(n=10, p=0.3), а не stats.binom.")
    sample = np.asarray(sample, dtype=float)
    if sample.ndim != 1 or sample.size == 0:
        raise ValueError("Первый аргумент — непустой одномерный массив наблюдений.")

    fig, ax = plt.subplots(figsize=(8, 4))
    lo = min(sample.min(), dist.ppf(0.001))
    hi = max(sample.max(), dist.ppf(0.999))
    if isinstance(dist.dist, stats.rv_discrete):
        values, counts = np.unique(sample, return_counts=True)
        ax.bar(values, counts / sample.size, width=0.8, color="#1F6BFF",
               alpha=0.35, label="частоты в выборке")
        ks = np.arange(np.floor(lo), np.ceil(hi) + 1)
        ax.vlines(ks, 0, dist.pmf(ks), color="#111827", lw=1.5)
        ax.plot(ks, dist.pmf(ks), "o", color="#111827", ms=5, label="pmf (теория)")
        ax.set_ylabel("вероятность / частота")
    else:
        ax.hist(sample, bins=50, density=True, color="#1F6BFF", alpha=0.35,
                label="гистограмма выборки")
        xs = np.linspace(lo, hi, 400)
        ax.plot(xs, dist.pdf(xs), color="#111827", lw=2, label="pdf (теория)")
        ax.set_ylabel("плотность")
    ax.set_xlabel("значение")
    ax.set_title(title or f"{dist.dist.name}: {sample.size} наблюдений")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()

## Ступень 1. По образцу

### Образец

Ниже всё уже написано. Прочитайте каждую строку и запустите. Предсказание — для каждой ячейки.

**Предсказание** (до запуска, словами): …

In [ ]:
X = stats.binom(n=10, p=0.3)           # биномиальный закон: 10 опытов, вероятность успеха 0.3
sample_x = X.rvs(size=10_000, random_state=1)
sample_x[:10]

**Предсказание** (до запуска, словами): …

In [ ]:
plot_dist(sample_x, X)

**Предсказание** (до запуска, словами): …

In [ ]:
print(f"E(X):   теория {X.mean():.3f}, выборка {sample_x.mean():.3f}")
print(f"Var(X): теория {X.var():.3f}, выборка {sample_x.var():.3f}")

### 1.1. Биномиальный закон

Создайте биномиальный закон `B` с $n=20$ и $p=0{,}25$. Получите выборку `sample_b` из 10 000 наблюдений
с `random_state=2`, нарисуйте её через `plot_dist` и выведите $E$ и $\mathrm{Var}$: теоретические и выборочные.
Замените `...` по образцу.

**Предсказание** (до запуска, словами): …

In [ ]:
B = ...
sample_b = ...
plot_dist(sample_b, B)
print(f"E(B):   теория {...:.3f}, выборка {...:.3f}")
print(f"Var(B): теория {...:.3f}, выборка {...:.3f}")

### 1.2. Нормальный закон

Создайте нормальный закон `N` с $E(N)=50$ и стандартным отклонением $\sigma=5$.
Остальное — как в 1.1, `random_state=3`. У нормального закона в SciPy параметры называются `loc` и `scale`.

**Предсказание** (до запуска, словами): …

In [ ]:
N = ...
sample_n = ...
plot_dist(sample_n, N)
print(f"E(N):   теория {...:.3f}, выборка {...:.3f}")
print(f"Var(N): теория {...:.3f}, выборка {...:.3f}")

### 1.3. Проверьте себя

Почему выборочные $E$ и $\mathrm{Var}$ близки к теоретическим, но не равны им?
Что изменится, если взять `random_state=4`? А если `size=100`?

**Ответ:** …

## Ступень 2. Сами выбираете и комбинируете

### 2.1. Несколько законов в словаре

Соберите четыре закона:

| Имя | Закон | Параметры |
|---|---|---|
| `"binom"` | биномиальный | $n=20$, $p=0{,}25$ |
| `"norm"` | нормальный | $E=50$, $\sigma=5$ |
| `"poisson"` | Пуассона | $\lambda=3$ |
| `"expon"` | показательный | интенсивность $\lambda=0{,}5$ |

Законы Пуассона и показательный найдите в документации [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html) сами:
как называются, какие у них параметры, как они связаны с $\lambda$. Проверьте себя: должно выйти $E=3$ и $E=2$.

**Структуры данных.**
- `params` — словарь: имя закона → словарь его параметров, например `"binom": {"n": 20, "p": 0.25}`.
- `laws` — словарь: имя закона → замороженное распределение, созданное из `params`.
- `samples` — словарь: имя закона → выборка из 10 000 наблюдений (`random_state=5`).

Для каждого закона выведите имя, теоретические и выборочные $E$ и $\mathrm{Var}$ и нарисуйте `plot_dist`.

**Предсказание** (до запуска, словами): …

In [ ]:
params = {
    "binom": {"n": 20, "p": 0.25},
    # допишите остальные три
}

### 2.2. Вероятность события: доля в выборке и теория

Для каждого события оцените вероятность **долей в выборке** (маской) и посчитайте её **методом закона** (`cdf` или `sf`).
Выведите оба числа рядом.

1. биномиальный: $P(X\ge 8)$;
2. нормальный: $P(45<X<55)$;
3. Пуассона: $P(X\le 1)$;
4. показательный: $P(X>4)$.

Перед кодом запишите, какой метод и с каким аргументом нужен в каждом случае. Осторожно с границей у дискретных законов.

**Методы:** …

**Предсказание** (до запуска, словами): …

### 2.3. Меняем параметр

Нарисуйте рядом (две ячейки или одну за другой): биномиальный закон с $n=20$ при $p=0{,}25$ и при $p=0{,}5$;
нормальный с $E=50$ при $\sigma=5$ и при $\sigma=10$. Для каждого — выборка и `plot_dist`.
Сначала предскажите, что сдвинется, а что станет шире.

**Предсказание** (до запуска, словами): …

Как изменились графики и почему? Свяжите с формулами $E$ и $\mathrm{Var}$.

**Ответ:** …

### 2.4. Почини

В каждой ячейке ровно одна ошибка. Запустите, прочитайте результат (или последнюю строку ошибки),
найдите ошибку и исправьте её **в той же ячейке**. Под ячейкой одной фразой запишите, в чём была ошибка.

**а)** Нужен нормальный закон с $E=50$ и **дисперсией** 25.

In [ ]:
W = stats.norm(loc=50, scale=25)
print("Var(W) =", W.var())

Ошибка: …

**б)** Нужна плотность стандартного нормального закона в точке 0,5.

In [ ]:
Z = stats.norm(loc=0, scale=1)
Z.pmf(0.5)

Ошибка: …

**в)** Нужно напечатать ожидание биномиального закона $n=20$, $p=0{,}25$.

In [ ]:
B = stats.binom(n=20, p=0.25)
print(f"E(B) = {B.mean}")

Ошибка: …

## Ступень 3. Задание с прошлого семинара: проверка партии

Это лаборатория 1 семинара 11, теперь в виде кода, который нужно дописать.
Опирайтесь на документацию и на вопросы преподавателю.

**Документация:** [binom](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binom.html),
[hypergeom](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.hypergeom.html).

**Условие.** В первой схеме независимо проверяют 10 изделий; каждое бракованное
с вероятностью 0,2. Во второй из партии 20 изделий, среди которых ровно 4
бракованных, случайно выбирают 10 без возвращения. $X$ — число бракованных
среди проверенных.

1. Создайте оба закона в словаре `batch_laws`.
2. Для каждого найдите $E(X)$, $\mathrm{Var}(X)$, $P(X\ge3)$ и 95-й перцентиль.
3. Получите выборки по 10 000 наблюдений (`random_state=11`) в словарь `batch_samples`,
   сравните частоту события $X\ge3$ с вероятностью и нарисуйте `plot_dist` для каждой схемы.
4. Ответьте на вопросы под кодом.

**Предсказание** (до запуска, словами): …

In [ ]:
batch_laws = {
    "с возвращением": ...,
    "без возвращения": ...,
}

for name, law in batch_laws.items():
    print(name)
    print("  E(X)        =", ...)
    print("  Var(X)      =", ...)
    print("  P(X >= 3)   =", ...)
    print("  95-й перцентиль =", ...)

**Предсказание** (до запуска, словами): …

In [ ]:
batch_samples = {}
for name, law in batch_laws.items():
    batch_samples[name] = ...

for name, law in batch_laws.items():
    s = batch_samples[name]
    print(f"{name}: частота X >= 3 = {...:.4f}, вероятность = {...:.4f}")
    plot_dist(s, law, title=name)

**Вопросы.**
1. Почему ожидания совпали, а дисперсии разные? Что делает выборка без возвращения с разбросом?
2. Чем отличаются частота в выборке и вероятность? Почему они близки?

**Ответ:** …